In [1]:
import numpy as np
import seaborn as sns
import pandas as pd
from matplotlib import pyplot as plt
from selenium.webdriver.support.expected_conditions import none_of

plt.rcParams['font.family']='Times New Roman,Microsoft YaHei'# 设置字体族，中文为微软雅黑，英文为Times New Roman
plt.rcParams['mathtext.fontset'] = 'stix' # 设置数%matplotlib qt学公式字体为stix
plt.style.use('seaborn-v0_8-paper')
# 设置全局参数
plt.rcParams['figure.facecolor'] = 'white'  # 设置图形的背景为透明
plt.rcParams['axes.facecolor'] = 'white'    # 设置轴域的背景为透明
plt.rcParams['savefig.facecolor'] = 'white' # 保存图像时背景透明
import matplotlib
# matplotlib.use('TkAgg')
%matplotlib inline
from sklearn.preprocessing import StandardScaler
import openchord as ocd
import pickle
from palettable.tableau import Tableau_10

import xml.etree.ElementTree as ET
import os

In [2]:

from music21 import *
us = environment.UserSettings()
us['musicxmlPath'] = r"E:\Learning Soft1\MS\bin\MuseScore3.exe"
us['musescoreDirectPNGPath'] = r"E:\Learning Soft1\MS\bin\MuseScore3.exe"
chroma_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
minor_chords = [s + 'm' for s in chroma_labels]  # 小三和弦
major_chords = chroma_labels  # 大三和弦
chord_labels = major_chords + minor_chords  # 共24个和弦

In [3]:
dfmusic = pd.read_pickle("df_chordkey,pkl")
dfbird = pd.read_pickle('final_birds_df.pkl')
dfbasic = pd.read_pickle("soundfeature_other.pkl")
# 去除重复列
dfbasic = dfbasic.loc[:, ~dfbasic.columns.duplicated()]


# 定义高度替换规则
def replace_level(row):
    if row['place'] == 'ZS':
        return '3 m' if row['level'] == 1 else f"{row['level']}m"
    elif row['place'] == 'JH':
        return {1: '1.5 m', 2: '4 m', 3: '8 m', 4: '14 m'}.get(row['level'], f"{row['level']}m")
    elif row['place'] == 'CM':
        return {1: '1.5 m', 2: '10 m', 3: '16 m', 4: '22 m'}.get(row['level'], f"{row['level']}m")
    return f"{row['level']}m"


# 替换 level 列的值
dfbasic['level'] = dfbasic.apply(replace_level, axis=1)
df = pd.merge(dfmusic, dfbird, on=['Date', 'place', 'level'], how='inner')
df = pd.merge(df, dfbasic, on=['Date', 'place', 'level'], how='inner')
# 栖息地分类映射
habitat_mapping = {
    'Aquatic and Wetland Surface Birds': ['Eurasian Wigeon', 'Tundra Swan', 'Mallard',
                                          'Greater White-fronted Goose', 'Common Goldeneye',
                                          'Green-winged Teal', 'Eurasian Coot', 'Eurasian Moorhen'],

    'Shoreline and Marsh Birds': ['Eurasian Curlew', 'Whimbrel'],

    'Grassland Ground Birds': ['Olive-backed Pipit', 'Yellow-browed Bunting', 'Yellow-billed Grosbeak'],

    'Shrub Layer Birds': ['Light-vented Bulbul', 'Chinese Hwamei', 'Japanese Tit',
                          'Silver-throated Tit', 'Chinese Blackbird', 'Pale-legged Leaf Warbler'],

    'Lower Canopy Birds': ['Pale Thrush', 'Oriental Magpie'],

    'Upper Canopy and Aerial Birds': ["Swinhoe's White-eye"]
}

# 将所有物种映射到栖息地类别
species_to_habitat = {}
for habitat, species_list in habitat_mapping.items():
    for species in species_list:
        species_to_habitat[species] = habitat


# 分类函数
def classify_bird_habitat(species):
    return species_to_habitat.get(species, 'Other')  # 若未匹配到则归类为 'Other'


# 应用分类函数
df['habitat'] = df['bird'].copy().apply(classify_bird_habitat)

# 根据分类映射顺序创建分类类型
habitat_order = list(habitat_mapping.keys())
df['habitat'] = pd.Categorical(df['habitat'], categories=habitat_order, ordered=True)
# 定义和弦质量的映射
chord_quality_map = {
    'maj': '',  # major 不需要后缀
    'min': 'm',  # minor 转换为 'm'
    'dim': 'dim',  # diminished 保持不变
    'aug': 'aug',  # augmented 保持不变
    '7': '7',  # dominant 7th
    'maj7': 'maj7',  # major 7th
    'min7': 'm7',  # minor 7th
    'dim7': 'dim7',  # diminished 7th
    'sus4': 'sus4',  # suspended 4th
    'sus2': 'sus2'  # suspended 2nd
}


def convert_chord_name(chord_name):
    if ':' in chord_name:
        root, quality = chord_name.split(':')
        # 使用映射字典转换和弦质量
        return f"{root}{chord_quality_map.get(quality, '')}"
    else:
        return chord_name


# 应用转换函数
df['event'] = df['event'].apply(convert_chord_name)



In [4]:
dfmusic2 = pd.read_pickle("music_var_chordkey2.pkl")
dfbasic2 = pd.read_pickle("soundfeature_other2.pkl")
# 去除重复列
dfbasic2 = dfbasic2.loc[:, ~dfbasic2.columns.duplicated()]
dfmusic2 = dfmusic2.loc[:, ~dfmusic2.columns.duplicated()]
df2 = pd.merge(dfmusic2, dfbasic2, on=['file'], how='left', suffixes=('', '_drop'))
df2 = df2[[col for col in df2.columns if not col.endswith('_drop')]]

# 设置24个调性名称
key_names = ['A major', 'Bb major', 'B major', 'C major', 'Db major',
              'D major', 'Eb major', 'E major', 'F major', 'F# major',
              'G major', 'Ab major', 'A minor', 'Bb minor', 'B minor',
              'C minor', 'C# minor', 'D minor', 'D# minor', 'E minor',
              'F minor', 'F# minor', 'G minor', 'G# minor']
# 转换 `key` 列
def parse_key(key_str):
    # 去掉大括号和空格，按逗号分割字符串
    key_str = key_str.strip('{}').replace(' ', '').split(',')
    # 将所有值转换为浮点数
    return np.array([float(k) for k in key_str])

# 应用转换函数
df2['Key'] = df2['Key'].apply(parse_key)
# 从每行的 Key 数组中选择置信度最大的 Key
df2['track'] = df2['Key'].apply(lambda x: key_names[np.argmax(x)])



# 栖息地分类映射
habitat_mapping2 = {
    'Aquatic and Wetland Surface Birds': ['Eurasian Wigeon', 'Tundra Swan', 'Mallard', 
                                          'Greater White-fronted Goose', 'Common Goldeneye', 
                                          'Green-winged Teal', 'Eurasian Coot', 'Eurasian Moorhen'],

    'Shoreline and Marsh Birds': ['Eurasian Curlew', 'Whimbrel'],

    'Grassland Ground Birds': ['Olive-backed Pipit', 'Yellow-browed Bunting', 'Yellow-billed Grosbeak'],

    'Shrub Layer Birds': ['Light-vented Bulbul', 'Chinese Hwamei', 'Japanese Tit', 
                          'Silver-throated Tit', 'Chinese Blackbird', 'Pale-legged Leaf Warbler'],

    'Lower Canopy Birds': ['Pale Thrush', 'Oriental Magpie'],

    'Upper Canopy and Aerial Birds': ["Swinhoe's White-eye"]
}

# 将所有物种映射到栖息地类别
species_to_habitat2 = {}
for habitat, species_list in habitat_mapping2.items():
    for species in species_list:
        species_to_habitat2[species] = habitat

# 分类函数
def classify_bird_habitat(species):
    return species_to_habitat2.get(species, 'Other')  # 若未匹配到则归类为 'Other'

bird_name_mapping = {
    'Eurasian Teal': 'Eurasian Moorhen',
    'Silver-throated Bushtit': 'Silver-throated Tit',
    'Common Moorhen': 'Eurasian Moorhen', 
    'Eurasian Whimbrel': 'Whimbrel'  
}

# 先转换 `birdenname`
df2['bird'] = df2['birdenname'].replace(bird_name_mapping)

# 应用分类函数
df2['habitat'] = df2['bird'].copy().apply(classify_bird_habitat)

# 根据分类映射顺序创建分类类型
df2['habitat'] = pd.Categorical(df2['habitat'], categories=habitat_order, ordered=True)

# 解析 chord 变量为列表
df2['chord'] = df2['chord'].str.strip('{}').str.split(',')
df2 = df2.explode('chord').rename(columns={'chord': 'event'})


In [7]:
len(list(df2['bird'].unique()))

20

In [5]:

def plotradarbar(name,df_normalized_results,outputname,habitat):
    
    if habitat =="global":
        dfplot=df_normalized_results.sort_values(by=name, ascending=False)
        
    else:
        selected_habitat = habitat 
        dfplot=df_normalized_results[df_normalized_results["habitat"]==selected_habitat].sort_values(by=name, ascending=False)
        
    # 示例数据（请替换为你的数据）
    variables = dfplot['chord'].values.tolist()
    values = dfplot[name].values.tolist()
    
    if name=='df_percentage':
        # min_val, max_val = np.min(values), np.max(values)  # 计算原始数据范围
        # normalized_values =(values - min_val) / (max_val - min_val) 
        # values2=normalized_values*150
        # max_threshold = values[1]*1.6  # 设定最大显示值，超过此值的柱子会被截断
        # clipped_values = np.clip(values, None, max_threshold)  # 截断柱子
        # NB=np.std(values)/np.mean(values)
        min_val, max_val = np.min(values), np.max(values)  # 计算原始数据范围
        normalized_values =(values - min_val) / (max_val - min_val) 
        max_threshold = values[1]*1.43  # 设定最大显示值，超过此值的柱子会被截断
        clipped_values = np.clip(values, None, max_threshold) 
        NB=1
    else:
        min_val, max_val = np.min(values), np.max(values)  # 计算原始数据范围
        normalized_values =(values - min_val) / (max_val - min_val) 
        clipped_values = values
        NB=1
        
    major_color = "#ee6123"  # 大三和弦颜色
    minor_color = "#61b3de"  # 小三和弦颜色
    colors = [major_color if chord in major_chords else minor_color for chord in variables]
    
    # 计算角度（保证均匀分布）
    theta = np.linspace(0, 2*np.pi, len(variables), endpoint=False)
    
    # 柱子的宽度和间隔
    widths = np.pi / (len(variables) * 1)  # 控制柱子宽度
    inner_radius =8*NB  # 设置空心半径
    
    # 创建极坐标图
    fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
    
    # 绘制柱子，确保它们从 `inner_radius` 开始，并加粗前 5 个和弦
    bars = ax.bar(theta, clipped_values, width=widths, bottom=inner_radius, color=colors,
                  edgecolor=["#a71930" if i < 5 else colors[i] for i in range(len(variables))], linewidth=2)
    # 单独设置前 5 个 bar 的 linestyle
    for i in range(5):
        bars[i].set_linestyle('--')  # 设置虚线边框
    
    
    # 设定最小半径，创造真正的空心效果
    ax.set_ylim(0,  20+ inner_radius)
    
    texts=[]
    # 添加变量标签（标上百分号）
    for i, (label, orig_value, clipped_value) in enumerate(zip(variables, values, clipped_values)):
        rotation = np.degrees(theta[i])  # 角度转换
        alignment = "center"
        text_rotation = rotation 
    
        # 如果截断，标注完整数值，但柱子高度受 max_threshold 限制
        
        text=ax.text(theta[i], clipped_value + inner_radius + 5 , 
                     f"{label}({orig_value:.1f}%)",
                     ha=alignment, va="center", fontsize=15 if name!='df_percentage' else 15, fontweight="bold",rotation= text_rotation,  rotation_mode="anchor")
        texts.append(text)
        
    if name=='df_percentage':
        # 添加截断标志 "//"
        for i, (orig_value, clipped_value) in enumerate(zip(values, clipped_values)):
            if orig_value > max_threshold:  # 只有超过 max_threshold 的才加截断标志
                ax.text(theta[i], clipped_value + inner_radius*0.7, "//", ha="center", va="center",
                        fontsize=16, fontweight="bold", color="black")
    
    
    
    # if name!='df_percentage':
    #     # 设置文本调整
    #     adjust_text(
    #         texts, 
    #         ax=ax, zorder=8)
    
    
    # 取消默认坐标轴
    ax.set_yticklabels([])
    ax.set_xticklabels([])
    ax.set_yticks([])
    ax.set_xticks([])
    ax.spines['polar'].set_visible(False)  # 去掉背景的圆圈
    
    # # 添加颜色图例
    # fig.subplots_adjust(bottom=0.2)
    # cbaxes = inset_axes(ax, width="100%", height="100%", loc="center",
    #                     bbox_to_anchor=(0.3, 0.05, 0.4, 0.02), bbox_transform=fig.transFigure)
    # norm = mpl.colors.Normalize(vmin=0, vmax=len(set(categories)))
    # cb = fig.colorbar(ScalarMappable(norm=norm, cmap=mpl.colors.ListedColormap(list(color_map.values()))),
    #                   cax=cbaxes, orientation="horizontal")
    # cb.set_label("Parameter Categories", size=12)
    # cb.set_ticks([])
    # plt.tight_layout()
    # **去掉背景**
    fig.patch.set_alpha(0)  # 整个 Figure 透明
    ax.set_facecolor('none')  # 坐标轴背景透明
    # plt.savefig(outputname, bbox_inches='tight', transparent=True)
    plt.savefig(outputname, transparent=True)
    # plt.show()
    plt.close(fig)



def remove_white_layers(input_svg, output_svg):
    # 解析 SVG 文件
    tree = ET.parse(input_svg)
    root = tree.getroot()
    namespace = "{http://www.w3.org/2000/svg}"  # SVG 命名空间
    
    # 遍历所有的 <g> 和 <rect> 元素，删除填充为白色的图层
    for elem in root.findall(f".//{namespace}g") + root.findall(f".//{namespace}rect"):
        fill_attr = elem.get("fill")
        if fill_attr and fill_attr.lower() in ["white", "#ffffff", "rgb(255,255,255)"]:
            root.remove(elem)  # 删除白色图层

    # 保存修改后的 SVG
    tree.write(output_svg)

def chordocd(habitat,transition_matrices_df, outputpath,thred=0.01):
    
    if habitat =="global":
        transition_df=transition_matrices_df
    else:
        transition_df=transition_matrices_df[habitat]
    # 获取转移概率矩阵和状态列表
    transition_probabilities = transition_df.copy()
    # Normalize the transition probability matrix
    transition_probabilities = transition_df.div(transition_df.sum(axis=1), axis=0)
    # 清理矩阵：将小于0.0001的值设置为0
    transition_probabilities[transition_probabilities < thred] = 0
    states = transition_probabilities.columns.tolist()
    # 生成和弦图
    fig_chord = ocd.Chord(transition_probabilities.values.tolist(), states)
    fig_chord.padding = 100
    fig_chord.font_size = 35
    fig_chord.font_family = "Times New Roman"
    fig_chord.colormap =  [
    "#D73027",  
    "#F46D43",  
    "#FDAE61",  
    "#FEE08B",  
    "#ABDDA4",  
    "#66C2A5",  
    "#3288BD",  
    "#5E4FA2",  
]
    fig_chord.font_weight = "bold"
    fig_chord.background = 'none'  # 让背景透明
    fig_chord.save_svg(outputpath)
    # 输入和输出文件
    input_svg = outputpath
    output_svg = outputpath
    # 执行删除操作
    remove_white_layers(input_svg, output_svg)

def musicscore(chord_data,habitat,outputdf,outputdf2):

    chroma_labels = ['C', 'C#', 'D', 'D#', 'E', 'F', 'F#', 'G', 'G#', 'A', 'A#', 'B']
    # 定义和弦音程
    chord_intervals = {
        'C': ['C', 'E', 'G'], 'C#': ['C#', 'F', 'G#'], 'D': ['D', 'F#', 'A'], 'D#': ['D#', 'G', 'A#'],
        'E': ['E', 'G#', 'B'], 'F': ['F', 'A', 'C'], 'F#': ['F#', 'A#', 'C#'], 'G': ['G', 'B', 'D'],
        'G#': ['G#', 'C', 'D#'], 'A': ['A', 'C#', 'E'], 'A#': ['A#', 'D', 'F'], 'B': ['B', 'D#', 'F#'],
    
        'Cm': ['C', 'E-', 'G'], 'C#m': ['C#', 'E', 'G#'], 'Dm': ['D', 'F', 'A'], 'D#m': ['D#', 'F#', 'A#'],
        'Em': ['E', 'G', 'B'], 'Fm': ['F', 'A-', 'C'], 'F#m': ['F#', 'A', 'C#'], 'Gm': ['G', 'B-', 'D'],
        'G#m': ['G#', 'B', 'D#'], 'Am': ['A', 'C', 'E'], 'A#m': ['A#', 'C#', 'F'], 'Bm': ['B', 'D', 'F#']
    }
    
    if habitat =="global":
        
        df2_chords_sorted = chord_data['df2_chords'].sort_values(ascending=False)
        df_chords_sorted = chord_data['df_chords'].sort_values(ascending=False)
        
    else:
        
        # 选择某个 habitat 下的 bird 进行五线谱绘制
        selected_habitat = habitat # 选择第一个栖息地
        
        # 获取 df2（鸟+环境） 和 df（鸟单独） 的前5种和弦，按出现频率排序
        df2_chords_sorted = chord_data[selected_habitat]['df2_chords'].sort_values(ascending=False)
        df_chords_sorted = chord_data[selected_habitat]['df_chords'].sort_values(ascending=False)
        
    # 只选前5个和弦，并转换为百分比
    df2_chords_sorted = df2_chords_sorted[df2_chords_sorted > 0.05].head(5) * 100
    df_chords_sorted = df_chords_sorted[df_chords_sorted > 0.05].head(5) * 100
    
    # 取两者有数据的和弦
    selected_chords = list(set(df2_chords_sorted.index).union(set(df_chords_sorted.index)))
    selected_chords = sorted(
        list(set(df2_chords_sorted.index).union(set(df_chords_sorted.index))),
        key=lambda x: df2_chords_sorted.get(x, df_chords_sorted.get(x, 0)), 
        reverse=True
    )
    
    # **左侧（df2）：鸟+环境**
    part_df2 = stream.Part()
    measure = stream.Measure(number=1)  # **每 5 个和弦一小节**
    count = 0
    
    
    from music21 import chord
    for label in selected_chords:
        if label not in chord_intervals:
            continue  # 跳过未知和弦
        if label not in df2_chords_sorted:
            continue  # df2 里没有的和弦不画
    
        # **生成和弦音符**
        notes = [pitch.Pitch(p) for p in chord_intervals[label]]
        for p in notes:
            p.octave = 5
    
        c = chord.Chord(notes)
        c.quarterLength = 1
        percentage = df2_chords_sorted[label]
        c.lyric = f"{label} ({percentage:.1f}%)"  # **在五线谱下方标注 Percentage**
        
        
    
        measure.append(c)
        count += 1
    
        # **每 5 个和弦换一小节**
        if count == 5:
            part_df2.append(measure)
            measure = stream.Measure(number=measure.number + 1)
            count = 0
        
        
    
    if len(measure) > 0:
        part_df2.append(measure)  # **添加最后一小节**
    
    
    
    # **右侧（df）：鸟单独鸣叫**
    part_df = stream.Part()
    measure = stream.Measure(number=1)
    count = 0
    
    
    for label in selected_chords:
        if label not in chord_intervals:
            continue
        if label not in df_chords_sorted:
            continue
    
        notes = [pitch.Pitch(p) for p in chord_intervals[label]]
        for p in notes:
            p.octave = 5
    
        c = chord.Chord(notes)
        c.quarterLength = 1
        percentage = df_chords_sorted[label]
        c.lyric = f"{label} ({percentage:.1f}%)"
    
        measure.append(c)
        count += 1
    
        if count == 5:
            part_df.append(measure)
            measure = stream.Measure(number=measure.number + 1)
            count = 0
    
    if len(measure) > 0:
        part_df.append(measure)
        

    
    # **创建五线谱**
    score = stream.Score()
    score.append(part_df2)
    score.write(fmt="musicxml.png", fp=outputdf2)
    
    
    
    score = stream.Score()
    score.append(part_df)
    score.write(fmt="musicxml.png", fp=outputdf)


In [6]:
# # 计算 df 和 df2 中每个鸟类的出现次数
# df_bird_counts = df['bird'].value_counts()
# df2_bird_counts = df2['bird'].value_counts()
# 
# # 计算 df['bird'] 的均值，筛选满足条件的鸟
# df_mean_bird_count = df_bird_counts.mean()-2*df_bird_counts.std()
# 
# # 统计 df2 中每个栖息地的鸟类数量
# bird_counts = df2.groupby(['habitat', 'bird']).size().reset_index(name='count')
# 
# # 选取每个 `habitat` 中 `df2` 里数量最多的 `bird`，但要求在 `df` 中的数量不低于均值
# top_birds = bird_counts.sort_values(by=['habitat', 'count'], ascending=[True, False])
# top_birds = top_birds[top_birds['bird'].isin(df_bird_counts[df_bird_counts > df_mean_bird_count].index)]
# top_birds = top_birds.drop_duplicates(subset=['habitat'], keep='first')

# 统计各个 habitat 选出的鸟的和弦信息
chord_data = {}
# 获取所有 unique 的 habitat
unique_habitats = df['habitat'].unique()

for habitat in unique_habitats:
    # 统计 df 和 df2 中该栖息地的和弦分布
    df_chords = df[df['habitat'] == habitat]['event'].value_counts(normalize=True)
    df2_chords = df2[df2['habitat'] == habitat]['event'].value_counts(normalize=True)
    


    # # 选出和弦最多的 5 种，并确保其比例 ≥ 5%
    # df_chords = df_chords[df_chords >= 0.05].nlargest(5)
    # df2_chords = df2_chords[df2_chords >= 0.05].nlargest(5)

    # 存储结果
    chord_data[habitat] = {'df_chords': df_chords, 'df2_chords': df2_chords}


In [7]:
# 计算各个 habitat 的归一化比例（百分比）并选出和弦
normalized_results = []

for habitat, data in chord_data.items():
    # 获取 df2（鸟+环境） 和 df（鸟单独）的前5种和弦，按出现频率排序
    df2_chords_sorted = data['df2_chords'].sort_values(ascending=False)
    df_chords_sorted = data['df_chords'].sort_values(ascending=False)

    # 取两者有数据的和弦
    selected_chords = list(set(df2_chords_sorted.index).union(set(df_chords_sorted.index)))

    # 计算归一化比例（转换为百分比）
    df2_chords_norm = (df2_chords_sorted / df2_chords_sorted.sum()) * 100
    df_chords_norm = (df_chords_sorted / df_chords_sorted.sum()) * 100

    # 存储结果
    for chord in selected_chords:
        normalized_results.append({
            'habitat': habitat,
            'chord': chord,
            'df_percentage': df_chords_norm.get(chord, 0),
            'df2_percentage': df2_chords_norm.get(chord, 0)
        })

# 转换为 DataFrame 便于查看
df_normalized_results = pd.DataFrame(normalized_results)



In [8]:

with open('transition_matrices_df.pkl', 'rb') as f:
    transition_matrices_df = pickle.load(f)
with open('transition_matrices_df2_0.01.pkl', 'rb') as f:
    transition_matrices_df2 = pickle.load(f)

In [9]:
import os

# 目标文件夹
output_dir = "globalandshanghai"

# 检查文件夹是否存在，不存在则创建
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 数据名称列表
name = ["df_percentage", "df2_percentage"]
dataname = ["shanghai","global"]
trans=[transition_matrices_df,transition_matrices_df2]
# 遍历 habitat_order 并执行操作
for habitat in habitat_order:
    for i in range(2):
        # 生成文件名
        ocd_filename = f"ocd{habitat}-{dataname[i]}.svg"
        radar_filename = f"radar{habitat}-{dataname[i]}.svg"
        
        # 执行操作并保存到指定目录
        chordocd(habitat, trans[i], os.path.join(output_dir, ocd_filename), thred=0.05)
        plotradarbar(name[i], df_normalized_results, os.path.join(output_dir, radar_filename),habitat)
    
    # 生成音乐五线谱文件名
    music_filename1 = os.path.join(output_dir, f"music{habitat}-{dataname[0]}.png")
    music_filename2 = os.path.join(output_dir, f"music{habitat}-{dataname[1]}.png")
    
    # 生成音乐五线谱
    musicscore(chord_data, habitat, music_filename1, music_filename2)

In [10]:
with open('global_transition_matrix_df.pkl', 'rb') as f:
    transition_matrices_df = pickle.load(f)
with open('global_transition_matrix_df2.pkl', 'rb') as f:
    transition_matrices_df2 = pickle.load(f)

In [11]:
# 统计各个 habitat 选出的鸟的和弦信息
chord_data = {}

df_chords = df['event'].value_counts(normalize=True)
df2_chords = df2['event'].value_counts(normalize=True)

# # 选出和弦最多的 5 种，并确保其比例 ≥ 5%
# df_chords = df_chords[df_chords >= 0.05].nlargest(5)
# df2_chords = df2_chords[df2_chords >= 0.05].nlargest(5)

# 存储结果
chord_data = {'df_chords': df_chords, 'df2_chords': df2_chords}

# 计算各个 habitat 的归一化比例（百分比）并选出和弦
normalized_results = []
df2_chords_sorted = chord_data['df2_chords'].sort_values(ascending=False)
df_chords_sorted = chord_data['df_chords'].sort_values(ascending=False)

# 取两者有数据的和弦
selected_chords = list(set(df2_chords_sorted.index).union(set(df_chords_sorted.index)))

# 计算归一化比例（转换为百分比）
df2_chords_norm = (df2_chords_sorted / df2_chords_sorted.sum()) * 100
df_chords_norm = (df_chords_sorted / df_chords_sorted.sum()) * 100

# 存储结果
for chord in selected_chords:
    normalized_results.append({
        'chord': chord,
        'df_percentage': df_chords_norm.get(chord, 0),
        'df2_percentage': df2_chords_norm.get(chord, 0)
    })

# 转换为 DataFrame 便于查看
df_normalized_results = pd.DataFrame(normalized_results)

In [12]:
import os

# 目标文件夹
output_dir = "globalandshanghai"

# 检查文件夹是否存在，不存在则创建
if not os.path.exists(output_dir):
    os.makedirs(output_dir)

# 数据名称列表
name = ["df_percentage", "df2_percentage"]
dataname = ["shanghai","global"]
trans=[transition_matrices_df,transition_matrices_df2]
for i in range(2):
    # 生成文件名
    ocd_filename = f"ocdglobal-{dataname[i]}.svg"
    radar_filename = f"radarglobal-{dataname[i]}.svg"
    
    # 执行操作并保存到指定目录
    chordocd("global", trans[i], os.path.join(output_dir, ocd_filename), thred=0.05)
    plotradarbar(name[i], df_normalized_results, os.path.join(output_dir, radar_filename), "global")

# 生成音乐五线谱文件名
music_filename1 = os.path.join(output_dir, f"musicglobal-{dataname[0]}.png")
music_filename2 = os.path.join(output_dir, f"musicglobal-{dataname[1]}.png")

# 生成音乐五线谱
musicscore(chord_data, "global", music_filename1, music_filename2)

In [13]:

# name='df_percentage'
# selected_habitat = list(chord_data.keys())[5]  
# dfplot=df_normalized_results[df_normalized_results["habitat"]==selected_habitat].sort_values(by=name, ascending=False)
# 
# # 示例数据（请替换为你的数据）
# variables = dfplot['chord'].values.tolist()
# values = dfplot[name].values.tolist()
# 
# if name=='df_percentage':
#     # min_val, max_val = np.min(values), np.max(values)  # 计算原始数据范围
#     # normalized_values =(values - min_val) / (max_val - min_val) 
#     # values2=normalized_values*150
#     # max_threshold = values[1]*1.6  # 设定最大显示值，超过此值的柱子会被截断
#     # clipped_values = np.clip(values, None, max_threshold)  # 截断柱子
#     # NB=np.std(values)/np.mean(values)
#     min_val, max_val = np.min(values), np.max(values)  # 计算原始数据范围
#     normalized_values =(values - min_val) / (max_val - min_val) 
#     max_threshold = values[1]*1.6  # 设定最大显示值，超过此值的柱子会被截断
#     clipped_values = np.clip(values, None, max_threshold) 
#     NB=1
# else:
#     min_val, max_val = np.min(values), np.max(values)  # 计算原始数据范围
#     normalized_values =(values - min_val) / (max_val - min_val) 
#     clipped_values = values
#     NB=1
# 
# major_color = "#ee6123"  # 大三和弦颜色
# minor_color = "#61b3de"  # 小三和弦颜色
# colors = [major_color if chord in major_chords else minor_color for chord in variables]
# 
# # 计算角度（保证均匀分布）
# theta = np.linspace(0, 2*np.pi, len(variables), endpoint=False)
# 
# # 柱子的宽度和间隔
# widths = np.pi / (len(variables) * 1)  # 控制柱子宽度
# inner_radius =4.5*NB  # 设置空心半径
# 
# # 创建极坐标图
# fig, ax = plt.subplots(figsize=(8, 8), subplot_kw={'projection': 'polar'})
# 
# # 绘制柱子，确保它们从 `inner_radius` 开始，并加粗前 5 个和弦
# bars = ax.bar(theta, clipped_values, width=widths, bottom=inner_radius, color=colors,
#               edgecolor=["#a71930" if i < 5 else colors[i] for i in range(len(variables))], linewidth=2)
# # 单独设置前 5 个 bar 的 linestyle
# for i in range(5):
#     bars[i].set_linestyle('--')  # 设置虚线边框
# 
# 
# # 设定最小半径，创造真正的空心效果
# ax.set_ylim(0, 10 + inner_radius + 2)
# 
# texts=[]
# # 添加变量标签（标上百分号）
# for i, (label, orig_value, clipped_value) in enumerate(zip(variables, values, clipped_values)):
#     rotation = np.degrees(theta[i])  # 角度转换
#     alignment = "center"
#     text_rotation = rotation 
# 
#     # 如果截断，标注完整数值，但柱子高度受 max_threshold 限制
# 
#     text=ax.text(theta[i], clipped_value + inner_radius*1.5, 
#                  f"{label}({orig_value:.1f}%)",
#                  ha=alignment, va="center", fontsize=13 if name!='df_percentage' else 13, fontweight="bold",rotation= text_rotation,  rotation_mode="anchor")
#     texts.append(text)
# 
# if name=='df_percentage':
#     # 添加截断标志 "//"
#     for i, (orig_value, clipped_value) in enumerate(zip(values, clipped_values)):
#         if orig_value > max_threshold:  # 只有超过 max_threshold 的才加截断标志
#             ax.text(theta[i], clipped_value + inner_radius - 2, "//", ha="center", va="center",
#                     fontsize=16, fontweight="bold", color="black")
# 
# 
# 
# # if name!='df_percentage':
# #     # 设置文本调整
# #     adjust_text(
# #         texts, 
# #         ax=ax, zorder=8)
# 
# 
# # 取消默认坐标轴
# ax.set_yticklabels([])
# ax.set_xticklabels([])
# ax.set_yticks([])
# ax.set_xticks([])
# ax.spines['polar'].set_visible(False)  # 去掉背景的圆圈
# 
# # # 添加颜色图例
# # fig.subplots_adjust(bottom=0.2)
# # cbaxes = inset_axes(ax, width="100%", height="100%", loc="center",
# #                     bbox_to_anchor=(0.3, 0.05, 0.4, 0.02), bbox_transform=fig.transFigure)
# # norm = mpl.colors.Normalize(vmin=0, vmax=len(set(categories)))
# # cb = fig.colorbar(ScalarMappable(norm=norm, cmap=mpl.colors.ListedColormap(list(color_map.values()))),
# #                   cax=cbaxes, orientation="horizontal")
# # cb.set_label("Parameter Categories", size=12)
# # cb.set_ticks([])
# # plt.tight_layout()
# # **去掉背景**
# fig.patch.set_alpha(0)  # 整个 Figure 透明
# ax.set_facecolor('none')  # 坐标轴背景透明
# plt.savefig("test.svg", bbox_inches='tight', dpi=300, transparent=True)
# 
# 
# plt.show()
